In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from mmidas.utils.tools import get_paths

%load_ext autoreload
%autoreload 2

In [2]:
toml_file = 'pyproject.toml'
config = get_paths(toml_file=toml_file)
data_path = config['paths']['main_dir'] / config['paths']['data_path'] / config['data']['anndata_file']
print(data_path)

/Users/yeganeh.marghi/github/LC-NE-MixRep/pyproject.toml
Getting files directories belong to data...
Getting files directories belong to models...
/Users/yeganeh.marghi/github/LC-NE-MixRep/data/Dbh_cluster.h5ad


In [3]:
scale_factor = 1e6
adata = sc.read_h5ad(data_path)
logcpm = adata.X.toarray()
gene_id = adata.var.index.values
print(f'All cells are normalized - {any(np.sum(np.exp(logcpm) - 1, axis=1) == scale_factor)}')
print(adata.obs.keys())

All cells are normalized - True
Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample_id', 'umi.counts',
       'batch', 'external_donor_name', 'sex', 'roi', 'rna_amplification_set',
       'library_prep_set', 'exclude', 'exclude2', 'load_name', 'doublet_score',
       'experiment', 'platform', 'percent.mt', 'percent.rb', 'cell_class'],
      dtype='object')


In [4]:
adata.obs['exclude'].unique(), adata.obs['exclude2'].unique(), sum(adata.obs['exclude2']=='YES')

(array(['No'], dtype=object), array(['No', 'YES'], dtype=object), 67)

In [5]:
adata.obs['batch'].unique(), (adata.obs['external_donor_name'].unique()), adata.obs['sex'].unique(), adata.obs['roi'].unique()

(array(['R8TX-24081301', 'R8TX-24081302', 'R8TX-24100104', 'R8TX-24100103'],
       dtype=object),
 array(['751451;751452;751453;751454;751455;751456;751457;751458;751459;751460',
        '751441;751442;751443;751444;751445;751446;751447;751448;751449;751450',
        '752994', '753005'], dtype=object),
 array(['F;F;F;F;F;F;F;F;F;F', 'M;M;M;M;M;M;M;M;M;M', 'M', 'F'],
       dtype=object),
 array(['Mouse 10x PONS - LC'], dtype=object))

In [ ]:
# Get highly variable genes
n_genes = 2000
genes_std = np.std(logcpm, axis=0)
hvgs_indx = np.argsort(genes_std)[::-1][:n_genes]

hvgs = pd.DataFrame({'gene': gene_id[hvgs_indx], 'std': genes_std[hvgs_indx]})
hvgs.to_csv(config['paths']['main_dir'] / config['paths']['data_path'] / 'HVGs_mmidas.csv', index=False)

print(f"Highly variable genes: {gene_id[hvgs_indx]}")

Highly variable genes: ['Xist' 'Uty' 'Zfp804b' ... 'Trp53inp2' 'Esrra' 'Tuba1b']


In [26]:
# load hvgs from file and compare with calculated ones
hvg_path = config['paths']['main_dir'] / config['paths']['data_path'] / config['data']['hvg_seurat']
hvgs_seurat = pd.read_csv(hvg_path, header=None).values.flatten()[1:]
print("Genes selected by Seurat:")
print(hvgs_seurat, len(hvgs_seurat))

# number of overlapping genes
n_overlap = len(set(gene_id[hvgs_indx]).intersection(set(hvgs_seurat)))
print(f'Number of overlapping genes: {n_overlap} ({n_overlap/n_genes*100:.2f}%)')

Genes selected by Seurat:
['Il1rapl2' 'Gm32647' 'Il1rapl1' ... 'Wdfy4' 'Tnni3k' 'Gm37853'] 2000
Number of overlapping genes: 403 (20.15%)


In [24]:
s_indx = np.array([np.where(gene_id==gg)[0][0] for gg in hvgs_seurat])
print(f'std of seurat genes: {np.sort(genes_std[s_indx])}')
print(f'std of calculated genes: {np.sort(genes_std[hvgs_indx])}')

std of seurat genes: [0.06605694 0.06844088 0.07168278 ... 2.55516076 2.63961755 3.3182655 ]
std of calculated genes: [1.71978452 1.71980385 1.71981075 ... 2.55516076 2.63961755 3.3182655 ]
